# Data Collection & Indicator calucation

In [1]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
import pandas as pd
import numpy as np
from vnstock import Vnstock
from datetime import datetime, timedelta

import pandas as pd
from datetime import datetime, timedelta
from vnstock import Vnstock
import yfinance as yf

def get_stock_data(symbol, years=3, interval="1D", lang="vi"):
    """
    Lấy dữ liệu tổng hợp về cổ phiếu:
      1. Dữ liệu giá (price history)
      2. Các chỉ tiêu tài chính (financial features)
      3. Các chỉ số kinh tế vĩ mô (economic indicators)

    symbol   : Mã cổ phiếu (VD: 'VCB')
    years    : Số năm cần lấy dữ liệu
    interval : Chu kỳ dữ liệu giá ('1D', '1W', '1M')
    lang     : Ngôn ngữ ('vi' hoặc 'en')

    Returns: dict chứa DataFrame cho từng nhóm dữ liệu
    """

    # === 1. Lấy dữ liệu giá ===
    end_date = datetime.today().strftime("%Y-%m-%d")
    start_date = (datetime.today() - timedelta(days=years * 365)).strftime("%Y-%m-%d")

    api = Vnstock()

    # Price history
    price_df = api.stock(symbol).quote.history(
        start=start_date, end=end_date, interval=interval
    )
    if 'time' in price_df.columns:
        price_df['time'] = pd.to_datetime(price_df['time'])
        price_df.set_index('time', inplace=True)

    # === 3. Macro features (VN indexes + commodities) ===
    vnindex = api.stock("VNINDEX").quote.history(start=start_date, end=end_date, interval=interval)
    vn30    = api.stock("VN30").quote.history(start=start_date, end=end_date, interval=interval)
    hnx     = api.stock("HNXINDEX").quote.history(start=start_date, end=end_date, interval=interval)

    vnindex = vnindex[['time','close']].rename(columns={'close':'vnindex'}).set_index(pd.to_datetime(vnindex['time'])).drop(columns=['time'])
    vn30 = vn30[['time','close']].rename(columns={'close':'vn30'}).set_index(pd.to_datetime(vn30['time'])).drop(columns=['time'])
    hnx = hnx[['time','close']].rename(columns={'close':'hnx'}).set_index(pd.to_datetime(hnx['time'])).drop(columns=['time'])

    # Global commodities (gold + oil via Yahoo Finance)
    gold = yf.download("GC=F", start=start_date, end=end_date, interval=str(interval).lower())
    gold.columns = ["Close", "High", "Low", "Open", "Volume"]
    gold = gold["Close"].rename("gold")
    
    oil = yf.download("BZ=F", start=start_date, end=end_date, interval=str(interval).lower())
    oil.columns = ["Close", "High", "Low", "Open", "Volume"]
    oil = oil["Close"].rename("oil")
    
    macro_df = pd.concat([vnindex, vn30, hnx, gold, oil], axis=1)

    '''
    return {
        "price": price_df,
        "macro": macro_df
    }
    '''
    return pd.concat([price_df, macro_df], axis='columns')

def add_calendar_features(df):
    """
    Adds calendar/categorical features exactly as described:
    - 12 months (one-hot)
    - 31 days (one-hot)
    - 5 weekdays (Mon-Fri, one-hot)
    - 6 hours (9-16, one-hot)
    - 4 minute segments (0,15,30,45, one-hot)
    - Monday morning indicator
    - Friday afternoon indicator
    - Pre-holiday afternoon indicator
    - Post-holiday morning indicator
    """
    df = df.copy()

    # --- Basic date/time features ---
    df['month'] = df.index.month
    df['day'] = df.index.day
    df['weekday'] = df.index.dayofweek  # Mon=0, Fri=4
    df['hour'] = df.index.hour
    df['minute'] = df.index.minute
    df['minute_segment'] = df['minute'] // 15  # 0,1,2,3 → 0,15,30,45

    # --- Monday morning & Friday afternoon ---
    df['monday_morning'] = ((df['weekday'] == 0) & (df['hour'] < 12)).astype(int)
    df['friday_afternoon'] = ((df['weekday'] == 4) & (df['hour'] >= 12)).astype(int)

    # --- Pre-holiday afternoon & Post-holiday morning ---
    # Detect holiday gaps (diff > 1 day)
    df['prev_day'] = df.index.to_series().shift(1)
    df['days_gap'] = (df.index.to_series() - df['prev_day']).dt.days.fillna(1)
    df['pre_holiday_afternoon'] = ((df['days_gap'] > 1) & (df['hour'] >= 12)).astype(int)
    df['post_holiday_morning'] = ((df['days_gap'] > 1) & (df['hour'] < 12)).astype(int)
    df.drop(columns=['prev_day','days_gap'], inplace=True)
    '''
    # --- One-hot encoding ---
    df = pd.get_dummies(df, columns=['month','day','weekday','hour','minute_segment'],
                        prefix=['M','D','W','H','MinSeg'], dtype=int)
    '''
    
    return df

data =  get_stock_data("VCB", years=20) # Lấy n năm gần nhất 
data.to_csv("Data Set/vcb_data.csv")

data = data.dropna()
data = add_calendar_features(data)

print(len(data.columns))
display(data)


## 👋 Chào mừng bạn đến với Vnstock!

Cảm ơn bạn đã sử dụng package phân tích chứng khoán #1 tại Việt Nam

* Tài liệu: [Sổ tay hướng dẫn](https://vnstocks.com/docs/category/s%E1%BB%95-tay-h%C6%B0%E1%BB%9Bng-d%E1%BA%ABn)
* Cộng đồng: [Nhóm Facebook](https://www.facebook.com/groups/vnstock.official)

Khám phá các tính năng mới nhất và tham gia cộng đồng để nhận hỗ trợ.
                

Phiên bản Vnai 2.1.9 đã có mặt, vui lòng cập nhật với câu lệnh : `pip install vnai --upgrade`.
Lịch sử phiên bản: https://pypi.org/project/vnai/#history
Phiên bản hiện tại 2.0.4

2025-09-09 14:05:26 - vnstock.common.data.data_explorer - INFO - Không phải là mã chứng khoán, thông tin công ty và tài chính không khả dụng.
2025-09-09 14:05:26 - vnstock.common.data.data_explorer - INFO - Không phải là mã chứng khoán, thông tin công ty và tài chính không khả dụng.
2025-09-09 14:05:27 - vnstock.common.data.data_explorer - INFO - Không phải là mã chứng khoán, thông tin công ty và tài chính không khả dụng.


YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


20


,open,high,low,close,volume,vnindex,vn30,hnx,gold,oil,month,day,weekday,hour,minute,minute_segment,monday_morning,friday_afternoon,pre_holiday_afternoon,post_holiday_morning
2012-02-06,5.59,5.59,5.50,5.55,404650.0,399.73,447.47,61.49,1722.800049,115.930000,2,6,0,0,0,0,1,0,0,0
2012-02-07,5.59,5.66,5.48,5.59,502100.0,401.08,449.31,62.53,1746.400024,116.230003,2,7,1,0,0,0,0,0,0,0
2012-02-08,5.59,5.82,5.59,5.82,801310.0,409.53,460.98,63.74,1729.300049,117.199997,2,8,2,0,0,0,0,0,0,0
2012-02-09,5.80,5.93,5.73,5.75,410610.0,411.39,464.73,63.82,1739.000000,118.589996,2,9,3,0,0,0,0,0,0,0
2012-02-10,5.71,5.75,5.62,5.62,924610.0,405.02,458.23,62.69,1723.300049,117.309998,2,10,4,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-08-29,69.20,70.10,68.50,68.60,14696100.0,1682.21,1865.38,279.98,3473.699951,68.120003,8,29,4,0,0,0,0,0,0,0
2025-09-03,68.80,68.80,67.00,67.00,14216900.0,1681.30,1859.59,282.70,3593.199951,67.599998,9,3,2,0,0,0,0,0,0,1
2025-09-04,67.50,69.00,66.90,68.90,8942100.0,1696.29,1883.59,283.99,3565.800049,66.989998,9,4,3,0,0,0,0,0,0,0
2025-09-05,69.50,69.60,67.50,67.50,11047800.0,1666.97,1845.48,280.67,3613.199951,65.500000,9,5,4,0,0,0,0,0,0,0


## Metrics Functions:

### Value-based

In [3]:
import numpy as np
import pandas as pd
import ta

def calculate_supertrend(df, period=10, multiplier=3):
    df = df.copy()
    hl2 = (df['High'] + df['Low']) / 2
    atr = ta.volatility.AverageTrueRange(df['High'], df['Low'], df['Close'], window=period).average_true_range()

    upperband = hl2 + (multiplier * atr)
    lowerband = hl2 - (multiplier * atr)

    supertrend = [None] * len(df)
    direction = [True] * len(df)

    for i in range(1, len(df)):
        if df['Close'][i] > upperband[i - 1]:
            direction[i] = True
        elif df['Close'][i] < lowerband[i - 1]:
            direction[i] = False
        else:
            direction[i] = direction[i - 1]
            if direction[i] and lowerband[i] < lowerband[i - 1]:
                lowerband[i] = lowerband[i - 1]
            if not direction[i] and upperband[i] > upperband[i - 1]:
                upperband[i] = upperband[i - 1]

        supertrend[i] = lowerband[i] if direction[i] else upperband[i]

    df['supertrend'] = supertrend
    df['supertrend_direction'] = np.array([int(i) for i in direction])
    return df

def hull_moving_average(series: pd.Series, window: int) -> pd.Series:
    """
    Calculate the Hull Moving Average (HMA) for a given price series.
    :param series: A pandas Series (typically closing prices).
    :param window: The HMA period window (e.g., 14).
    :return: A pandas Series of the HMA values.
    """
    if window < 1:
        raise ValueError("Window must be a positive integer")

    half_length = int(window / 2)
    sqrt_length = int(np.sqrt(window))

    wma_half = series.rolling(window=half_length).mean()
    wma_full = series.rolling(window=window).mean()
    diff = 2 * wma_half - wma_full
    hma = diff.rolling(window=sqrt_length).mean()

    return hma

def detrended_price_oscillator(series: pd.Series, window: int = 14) -> pd.Series:
    """
    Calculate Detrended Price Oscillator (DPO).
    DPO = Price - SMA(Price, window)
    SMA is shifted back by (window / 2 + 1)
    """
    sma = series.rolling(window=window).mean()
    shift = int((window / 2) + 1)
    dpo = series.shift(shift) - sma
    return dpo

def calculate_value_based_indicators(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Rolling 3-day features
    df['rolling_mean_3'] = df['Close'].rolling(window=3).mean()
    df['rolling_std_3'] = df['Close'].rolling(window=3).std()
    df['rolling_return_3'] = df['Close'].pct_change(periods=3)

    # Rolling z-score (custom)
    window = 5
    rolling_mean = df['Close'].rolling(window).mean()
    rolling_std = df['Close'].rolling(window).std()
    df['zscore_5'] = (df['Close'] - rolling_mean) / rolling_std


    # RSI
    df['rsi'] = ta.momentum.RSIIndicator(close=df['Close']).rsi()

    # MACD
    macd = ta.trend.MACD(close=df['Close'])
    df['macd'] = macd.macd()
    df['macd_signal'] = macd.macd_signal()
    df['macd_diff'] = macd.macd_diff()

    df['macd_zscore'] = (df['macd'] - df['macd'].rolling(20).mean()) / df['macd'].rolling(20).std()
    
    # Bollinger Bands
    bb = ta.volatility.BollingerBands(close=df['Close'])
    df['bb_upper'] = bb.bollinger_hband()
    df['bb_middle'] = bb.bollinger_mavg()
    df['bb_lower'] = bb.bollinger_lband()

    # EMA and SMA
    df['ema_12'] = ta.trend.EMAIndicator(close=df['Close'], window=12).ema_indicator()
    df['ema_26'] = ta.trend.EMAIndicator(close=df['Close'], window=26).ema_indicator()
    df['sma_20'] = ta.trend.SMAIndicator(close=df['Close'], window=20).sma_indicator()
    df['sma_50'] = ta.trend.SMAIndicator(close=df['Close'], window=50).sma_indicator()

    # Stochastic Oscillator
    stoch = ta.momentum.StochasticOscillator(high=df['High'], low=df['Low'], close=df['Close'])
    df['stoch_k'] = stoch.stoch()
    df['stoch_d'] = stoch.stoch_signal()

    # ATR
    df['atr'] = ta.volatility.AverageTrueRange(high=df['High'], low=df['Low'], close=df['Close']).average_true_range()

    # VWAP
    df['vwap'] = ta.volume.VolumeWeightedAveragePrice(high=df['High'], low=df['Low'], close=df['Close'], volume=df['Volume']).volume_weighted_average_price()

    # Ichimoku Cloud (fixed)
    ichimoku = ta.trend.IchimokuIndicator(high=df['High'], low=df['Low'])
    df['ichimoku_a'] = ichimoku.ichimoku_a()
    df['ichimoku_b'] = ichimoku.ichimoku_b()
    df['ichimoku_base_line'] = ichimoku.ichimoku_base_line()

    # Keltner Channels
    kc = ta.volatility.KeltnerChannel(close=df['Close'], high=df['High'], low=df['Low'])
    df['kc_upper'] = kc.keltner_channel_hband()
    df['kc_middle'] = kc.keltner_channel_mband()
    df['kc_lower'] = kc.keltner_channel_lband()

    # Chaikin Money Flow (CMF)
    df['cmf'] = ta.volume.ChaikinMoneyFlowIndicator(high=df['High'], low=df['Low'], close=df['Close'], volume=df['Volume']).chaikin_money_flow()

    # Donchian Channels (manual)
    df['donchian_upper'] = df['High'].rolling(window=20).max()
    df['donchian_lower'] = df['Low'].rolling(window=20).min()
    df['donchian_middle'] = (df['donchian_upper'] + df['donchian_lower']) / 2

    # CCI
    df['cci'] = ta.trend.CCIIndicator(high=df['High'], low=df['Low'], close=df['Close']).cci()

    # OBV
    df['obv'] = ta.volume.OnBalanceVolumeIndicator(close=df['Close'], volume=df['Volume']).on_balance_volume()

    # Williams %R
    df['williams_r'] = ta.momentum.WilliamsRIndicator(high=df['High'], low=df['Low'], close=df['Close']).williams_r()

    # Supertrend (manual)
    df = calculate_supertrend(df)

    # Hull MA
    df['hma'] = hull_moving_average(df['Close'], window=14)

    # DPO
    df['dpo'] = detrended_price_oscillator(df['Close'], window=14)

    return df

### Rules-based

In [4]:
import pandas as pd

def calculate_rule_based_indicators(df: pd.DataFrame) -> pd.DataFrame:
    df = calculate_value_based_indicators(df)

    df['rsi_overbought'] = (df['rsi'] > 70).astype(int)
    df['rsi_oversold'] = (df['rsi'] < 30).astype(int)

    df['macd_cross'] = ((df['macd'] > df['macd_signal']) & (df['macd'].shift(1) <= df['macd_signal'].shift(1))).astype(int)

    df['price_breaks_bb_upper'] = (df['Close'] > df['bb_upper']).astype(int)
    df['price_breaks_bb_lower'] = (df['Close'] < df['bb_lower']).astype(int)

    df['price_above_ema12'] = (df['Close'] > df['ema_12']).astype(int)
    df['price_above_ema26'] = (df['Close'] > df['ema_26']).astype(int)
    df['price_above_sma20'] = (df['Close'] > df['sma_20']).astype(int)

    df['stoch_overbought'] = (df['stoch_k'] > 80).astype(int)
    df['stoch_oversold'] = (df['stoch_k'] < 20).astype(int)

    df['atr_high_volatility'] = (df['atr'] > df['atr'].rolling(20).mean()).astype(int)

    df['price_above_vwap'] = (df['Close'] > df['vwap']).astype(int)

    df['ichimoku_bullish'] = (df['Close'] > df['ichimoku_a']).astype(int)
    df['ichimoku_bearish'] = (df['Close'] < df['ichimoku_b']).astype(int)

    df['kc_bullish'] = (df['Close'] > df['kc_upper']).astype(int)
    df['kc_bearish'] = (df['Close'] < df['kc_lower']).astype(int)

    df['cmf_bullish'] = (df['cmf'] > 0).astype(int)
    df['cmf_bearish'] = (df['cmf'] < 0).astype(int)

    df['donchian_bullish'] = (df['Close'] > df['donchian_upper']).astype(int)
    df['donchian_bearish'] = (df['Close'] < df['donchian_lower']).astype(int)

    df['cci_bullish'] = (df['cci'] > 100).astype(int)
    df['cci_bearish'] = (df['cci'] < -100).astype(int)

    df['obv_bullish'] = (df['obv'] > df['obv'].shift(1)).astype(int)

    df['williams_r_overbought'] = (df['williams_r'] > -20).astype(int)
    df['williams_r_oversold'] = (df['williams_r'] < -80).astype(int)

    df['supertrend_bullish'] = df['supertrend_direction'].astype(int)

    df['hma_bullish'] = (df['Close'] > df['hma']).astype(int)
    df['hma_bearish'] = (df['Close'] < df['hma']).astype(int)

    df['dpo_bullish'] = (df['dpo'] > 0).astype(int)
    df['dpo_bearish'] = (df['dpo'] < 0).astype(int)

    return df

### Package for calculation:

In [5]:
import pandas as pd
import ta

def generate_trading_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Generate a strong trading feature set using the 'ta' library.
    Input: DataFrame with ['open', 'high', 'low', 'close', 'volume']
    Output: DataFrame with added features
    """
    df = df.copy()

    # Drop rows with NaNs after indicator calculation
    df = ta.add_all_ta_features(
        df,
        open="open",
        high="high",
        low="low",
        close="close",  
        volume="volume",
        fillna=True
    )

    return df

## Data Preparation:

### Defaut:

In [6]:
training_data = data.copy()
'''
training_data["return"] = training_data['close'].diff()
training_data["log_return"] = np.log(training_data['close']).diff()
'''
training_data["return"] = training_data['close'] - training_data['open']
training_data["log_return"] = np.log(training_data['close']) - np.log(training_data['open'])

training_data = generate_trading_features(df= training_data)

### shift for lagging features:

In [7]:
# Dictionary: feature name lists mapped to their required shift periods
shift_dict = {
    1: [
        'volume_adi', 'volume_obv', 'volume_vpt', 'volume_nvi', 'volume_vwap',
        'trend_psar_up', 'trend_psar_down', 'trend_psar_up_indicator', 'trend_psar_down_indicator',
        'others_cr',
    ],
    10: ['momentum_kama'],
    12: ['momentum_roc'],
    13: ['volume_fi'],
    14: [
        'volume_em', 'volume_sma_em', 'volume_mfi',
        'volatility_atr',
        'momentum_rsi', 'momentum_stoch_rsi', 'momentum_stoch_rsi_k', 'momentum_stoch_rsi_d',
        'momentum_stoch', 'momentum_stoch_signal', 'momentum_wr',
        'trend_adx', 'trend_adx_pos', 'trend_adx_neg',
        'trend_vortex_ind_pos', 'trend_vortex_ind_neg', 'trend_vortex_ind_diff',
    ],
    15: ['trend_trix'],
    20: [
        'volatility_bbm', 'volatility_bbh', 'volatility_bbl', 'volatility_bbw', 'volatility_bbp',
        'volatility_bbhi', 'volatility_bbli',
        'volatility_kcc', 'volatility_kch', 'volatility_kcl', 'volatility_kcw', 'volatility_kcp',
        'volatility_kchi', 'volatility_kcli',
        'volatility_dcl', 'volatility_dch', 'volatility_dcm', 'volatility_dcw', 'volatility_dcp',
        'trend_sma_fast', 'trend_dpo', 'trend_cci',
        'others_dr', 'others_dlr', 'trend_aroon_up', 'trend_aroon_down', 'trend_aroon_ind',
        'momentum_tsi',
    ],
    25: ['trend_mass_index'],
    26: ['trend_ema_slow', 'trend_ichimoku_base', 'trend_ichimoku_a', 'trend_visual_ichimoku_a'],
    30: ['trend_kst', 'trend_kst_sig', 'trend_kst_diff'],
    34: ['momentum_ao'],
    35: [
        'trend_macd', 'trend_macd_signal', 'trend_macd_diff',
        'momentum_ppo', 'momentum_ppo_signal', 'momentum_ppo_hist',
        'momentum_pvo', 'momentum_pvo_signal', 'momentum_pvo_hist',
    ],
    52: ['trend_ichimoku_b', 'trend_visual_ichimoku_b']
}

# Shift lagging features in training_data
for shift_period, feature_list in shift_dict.items():
    for feature in feature_list:
        training_data[feature] = training_data[feature].shift(shift_period)
        
training_data = training_data.dropna()

In [8]:
display(training_data)
display(training_data.info())

,open,high,low,close,volume,vnindex,vn30,hnx,gold,oil,...,momentum_ppo,momentum_ppo_signal,momentum_ppo_hist,momentum_pvo,momentum_pvo_signal,momentum_pvo_hist,momentum_kama,others_dr,others_dlr,others_cr
2012-04-23,7.82,7.91,7.79,7.82,404510.0,465.17,533.60,77.81,1631.900024,118.709999,...,4.764203,3.463147,1.301056,18.878575,16.927344,1.951231,6.833936,0.704225,0.701757,41.261261
2012-04-24,7.84,7.84,7.70,7.77,643880.0,465.65,534.89,78.57,1643.000000,118.160004,...,5.102715,3.791061,1.311654,16.605561,16.862987,-0.257426,6.836437,-1.398601,-1.408474,40.900901
2012-04-25,7.89,7.89,7.72,7.77,560280.0,472.87,542.78,79.55,1641.400024,119.120003,...,5.701212,4.173091,1.528121,22.493083,17.989006,4.504076,6.837802,-0.283688,-0.284091,40.000000
2012-04-26,7.84,7.84,7.74,7.74,507420.0,470.21,538.10,78.74,1659.599976,119.919998,...,5.662003,4.470873,1.191129,22.381355,18.867476,3.513879,6.945705,-3.413940,-3.473576,40.000000
2012-04-27,7.86,7.96,7.77,7.86,587740.0,473.77,541.20,79.86,1664.000000,119.830002,...,5.561200,4.688938,0.872261,20.622134,19.218408,1.403726,7.046370,1.767305,1.751870,39.459459
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-08-29,69.20,70.10,68.50,68.60,14696100.0,1682.21,1865.38,279.98,3473.699951,68.120003,...,1.900602,0.864885,1.035717,29.479506,18.393501,11.086005,62.726694,0.000000,0.000000,1143.243243
2025-09-03,68.80,68.80,67.00,67.00,14216900.0,1681.30,1859.59,282.70,3593.199951,67.599998,...,2.156363,1.123181,1.033182,24.856416,19.686084,5.170332,62.933113,1.495017,1.483951,1136.036036
2025-09-04,67.50,69.00,66.90,68.90,8942100.0,1696.29,1883.59,283.99,3565.800049,66.989998,...,2.196037,1.337752,0.858285,20.790429,19.906953,0.883476,63.037632,0.654664,0.652531,1107.207207
2025-09-05,69.50,69.60,67.50,67.50,11047800.0,1666.97,1845.48,280.67,3613.199951,65.500000,...,2.253813,1.520964,0.732849,18.659654,19.657493,-0.997839,63.037097,1.300813,1.292425,1141.441441


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 3215 entries, 2012-04-23 to 2025-09-08
Columns: 108 entries, open to others_cr
dtypes: float64(98), int32(6), int64(4)
memory usage: 2.6 MB


None

### Classification target:

In [9]:
#classification_target= pd.Series( np.where(training_data["close"].shift(-1) > training_data["close"], 1, 0), index= training_data.index)

classification_target= pd.Series( np.where(training_data["close"] > training_data["open"], 1, 0), index= training_data.index)

print(classification_target)

2012-04-23    0
2012-04-24    0
2012-04-25    0
2012-04-26    0
2012-04-27    0
             ..
2025-08-29    0
2025-09-03    0
2025-09-04    1
2025-09-05    0
2025-09-08    0
Length: 3215, dtype: int64


In [10]:
len(classification_target.loc[ classification_target==1 ]) / len(classification_target)

0.44261275272161743

### Strength/ Confidence target

In [11]:
'''
returns = training_data['close'].diff()
n= 10

vol = returns.rolling(window=n).std()
display(vol)

strength = np.where( vol < 1, 0, np.where(vol < 2.5, 1, 2) )
count= np.unique(strength, return_counts=True)
print(count)
for i, j in zip(count[0], count[1]):
    print(f"{i}: {j / count[1].sum()}")
    
display(pd.concat([pd.DataFrame(data=vol.values, index=vol.index, columns=["Volatility"]), pd.DataFrame(strength, index=vol.index, columns=['Strength'])], axis='columns'))
'''

'\nreturns = training_data[\'close\'].diff()\nn= 10\n\nvol = returns.rolling(window=n).std()\ndisplay(vol)\n\nstrength = np.where( vol < 1, 0, np.where(vol < 2.5, 1, 2) )\ncount= np.unique(strength, return_counts=True)\nprint(count)\nfor i, j in zip(count[0], count[1]):\n    print(f"{i}: {j / count[1].sum()}")\n    \ndisplay(pd.concat([pd.DataFrame(data=vol.values, index=vol.index, columns=["Volatility"]), pd.DataFrame(strength, index=vol.index, columns=[\'Strength\'])], axis=\'columns\'))\n'

In [12]:
"""
strength_target= pd.Series( strength, index=vol.index )
strength_target
"""

'\nstrength_target= pd.Series( strength, index=vol.index )\nstrength_target\n'

### Regression target

In [13]:
'''
regression_target= pd.Series( np.where(data["Close"] > data["Open"], 1, 0), index= data.index)[-len(training_data):]
len(regression_target)
'''

'\nregression_target= pd.Series( np.where(data["Close"] > data["Open"], 1, 0), index= data.index)[-len(training_data):]\nlen(regression_target)\n'

# File Saving:

In [14]:
training_data["target"] = classification_target
training_data["target"] = training_data["target"].shift(-1)

#training_data["strength target"] =  strength_target
#training_data["regression target"] = regression_target

training_data = training_data.dropna()
training_data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 3214 entries, 2012-04-23 to 2025-09-05
Columns: 109 entries, open to target
dtypes: float64(99), int32(6), int64(4)
memory usage: 2.6 MB


In [15]:
training_data

,open,high,low,close,volume,vnindex,vn30,hnx,gold,oil,...,momentum_ppo_signal,momentum_ppo_hist,momentum_pvo,momentum_pvo_signal,momentum_pvo_hist,momentum_kama,others_dr,others_dlr,others_cr,target
2012-04-23,7.82,7.91,7.79,7.82,404510.0,465.17,533.60,77.81,1631.900024,118.709999,...,3.463147,1.301056,18.878575,16.927344,1.951231,6.833936,0.704225,0.701757,41.261261,0.0
2012-04-24,7.84,7.84,7.70,7.77,643880.0,465.65,534.89,78.57,1643.000000,118.160004,...,3.791061,1.311654,16.605561,16.862987,-0.257426,6.836437,-1.398601,-1.408474,40.900901,0.0
2012-04-25,7.89,7.89,7.72,7.77,560280.0,472.87,542.78,79.55,1641.400024,119.120003,...,4.173091,1.528121,22.493083,17.989006,4.504076,6.837802,-0.283688,-0.284091,40.000000,0.0
2012-04-26,7.84,7.84,7.74,7.74,507420.0,470.21,538.10,78.74,1659.599976,119.919998,...,4.470873,1.191129,22.381355,18.867476,3.513879,6.945705,-3.413940,-3.473576,40.000000,0.0
2012-04-27,7.86,7.96,7.77,7.86,587740.0,473.77,541.20,79.86,1664.000000,119.830002,...,4.688938,0.872261,20.622134,19.218408,1.403726,7.046370,1.767305,1.751870,39.459459,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-08-28,69.50,70.60,68.70,69.00,8799400.0,1680.86,1861.20,276.63,3431.800049,68.620003,...,0.605956,0.882843,27.378349,15.621999,11.756349,62.478224,-1.149425,-1.156082,1145.045045,0.0
2025-08-29,69.20,70.10,68.50,68.60,14696100.0,1682.21,1865.38,279.98,3473.699951,68.120003,...,0.864885,1.035717,29.479506,18.393501,11.086005,62.726694,0.000000,0.000000,1143.243243,0.0
2025-09-03,68.80,68.80,67.00,67.00,14216900.0,1681.30,1859.59,282.70,3593.199951,67.599998,...,1.123181,1.033182,24.856416,19.686084,5.170332,62.933113,1.495017,1.483951,1136.036036,1.0
2025-09-04,67.50,69.00,66.90,68.90,8942100.0,1696.29,1883.59,283.99,3565.800049,66.989998,...,1.337752,0.858285,20.790429,19.906953,0.883476,63.037632,0.654664,0.652531,1107.207207,0.0


In [16]:
training_data.to_csv("Data Set/vcb_Training_Data.csv")

# Convergent Testing

In [17]:
'''
df1 = data.copy()
df1 = generate_trading_features(df= df1)
'''
df1 = training_data.copy()

'''
df1['target'] = pd.Series( np.where(df1["close"].shift(-1) > df1["close"], 1, 0), index= df1.index)
'''
df1['target'] = pd.Series( np.where(df1["return"].shift(-1) > 0, 1, 0), index= df1.index)
arr = df1.corrwith(df1.target)

print("Corr:")
print( arr.loc[arr > 0].sort_values(ascending=False) )

print("Var:")
print( df1.var() )

Corr:
target                   1.000000
volume_cmf               0.058229
momentum_stoch_signal    0.035596
momentum_uo              0.032971
momentum_wr              0.031320
momentum_stoch           0.031320
trend_vortex_ind_diff    0.030570
trend_vortex_ind_pos     0.029875
momentum_stoch_rsi_d     0.029531
volume_nvi               0.026822
momentum_roc             0.026794
trend_aroon_down         0.025724
volume_fi                0.023042
volatility_ui            0.022131
momentum_stoch_rsi_k     0.021862
volume_adi               0.021712
monday_morning           0.020832
others_dlr               0.018132
others_dr                0.017939
momentum_rsi             0.015841
trend_adx_pos            0.015306
volume_mfi               0.014742
volatility_bbw           0.013620
post_holiday_morning     0.012128
momentum_stoch_rsi       0.010576
volume_em                0.009914
volatility_kcw           0.008793
volume_sma_em            0.007312
trend_adx                0.004164
trend_ks

In [18]:
df1 = data.copy()
df1 = generate_trading_features(df= df1)

# Shift lagging features in training_data
for shift_period, feature_list in shift_dict.items():
    for feature in feature_list:
        df1[feature] = df1[feature].shift(shift_period)
df1 = df1.dropna()
display(df1.head())

df1['target'] = pd.Series( np.where(df1["close"].shift(-1) > df1["close"], 1, 0), index= df1.index)
arr = df1.corrwith(df1.target)

arr.loc[arr > 0].sort_values(ascending=False)

,open,high,low,close,volume,vnindex,vn30,hnx,gold,oil,...,momentum_ppo,momentum_ppo_signal,momentum_ppo_hist,momentum_pvo,momentum_pvo_signal,momentum_pvo_hist,momentum_kama,others_dr,others_dlr,others_cr
2012-04-23,7.82,7.91,7.79,7.82,404510.0,465.17,533.60,77.81,1631.900024,118.709999,...,4.764203,3.463147,1.301056,18.878575,16.927344,1.951231,6.833936,0.704225,0.701757,41.261261
2012-04-24,7.84,7.84,7.70,7.77,643880.0,465.65,534.89,78.57,1643.000000,118.160004,...,5.102715,3.791061,1.311654,16.605561,16.862987,-0.257426,6.836437,-1.398601,-1.408474,40.900901
2012-04-25,7.89,7.89,7.72,7.77,560280.0,472.87,542.78,79.55,1641.400024,119.120003,...,5.701212,4.173091,1.528121,22.493083,17.989006,4.504076,6.837802,-0.283688,-0.284091,40.000000
2012-04-26,7.84,7.84,7.74,7.74,507420.0,470.21,538.10,78.74,1659.599976,119.919998,...,5.662003,4.470873,1.191129,22.381355,18.867476,3.513879,6.945705,-3.413940,-3.473576,40.000000
2012-04-27,7.86,7.96,7.77,7.86,587740.0,473.77,541.20,79.86,1664.000000,119.830002,...,5.561200,4.688938,0.872261,20.622134,19.218408,1.403726,7.046370,1.767305,1.751870,39.459459


target                   1.000000
trend_macd_diff          0.040596
volatility_ui            0.035131
others_dlr               0.033893
others_dr                0.033434
trend_vortex_ind_pos     0.025107
trend_vortex_ind_diff    0.025053
momentum_stoch_rsi_d     0.024853
momentum_ppo_hist        0.023833
trend_aroon_down         0.022334
momentum_stoch_signal    0.021119
monday_morning           0.019158
volume_fi                0.018416
trend_kst_diff           0.017220
trend_adx                0.015581
trend_aroon_up           0.015563
post_holiday_morning     0.015365
trend_adx_pos            0.014937
volume_sma_em            0.013149
momentum_stoch           0.012337
momentum_wr              0.012337
momentum_roc             0.011868
volatility_kcp           0.011023
volatility_bbp           0.010739
momentum_stoch_rsi_k     0.010680
momentum_rsi             0.008196
volume_vpt               0.004046
trend_trix               0.003731
trend_cci                0.003719
volume_em     

In [19]:
returns = training_data['return'].to_numpy()

total_return = 0
for i in returns:
    total_return += i if i > 0 else 0

total_return

555.8300000000002

In [20]:
import joblib

model_infor = joblib.load("/Users/sonlamtong/Desktop/Python/Thesis/Technical Analysis /ML models/prediction dataset/Classification/knn_model_with_threshold.pkl")
model_infor

{'model': KNeighborsClassifier(algorithm='kd_tree', leaf_size=16, n_jobs=-1,
                      n_neighbors=3, p=1),
 'threshold': 0.0,
 'feature set': ['month', 'weekday', 'monday_morning', 'return', 'log_return']}

In [21]:
from sklearn.metrics import f1_score

df_1 = training_data.loc[:, ~training_data.columns.isin(["target", "strength target", "regression target"])].copy()
y = training_data.target.copy()
model = model_infor["model"]

model.fit(df_1, y)
y_pred = model.predict(df_1)

f1 = f1_score(y_pred=y_pred, y_true=y)

print(f1, total_return)

0.7235742826007991 555.8300000000002


In [22]:
target = training_data['target']

len(target.to_numpy()), len(classification_target)

(3214, 3215)

In [23]:
record_1 = []
signals = []
model_return = 0
for i in range(len(y_pred)-1):
    if y_pred[i] == 1: ## only consider long trades with TP(+return) and FP(-return)
        model_return += abs(returns[i+1]) if y_pred[i] == target.to_numpy()[i] else -abs(returns[i+1])
        signals.append( (target.to_numpy()[i], returns[i+1]) )

        record_1.append(abs(returns[i+1]) if y_pred[i] == target.to_numpy()[i] else -abs(returns[i+1]))
print(model_return)
print(model_return / total_return )

305.04999999999984
0.5488188834715646


In [24]:
record_2 = []
model_return = 0
for i in range(len(y_pred)-1):
    if y_pred[i] == 1:
        model_return += returns[i+1]
        record_2.append(returns[i+1])
print(model_return)
print(model_return / total_return )

305.04999999999984
0.5488188834715646


In [25]:
check = []
for i in range(len(y_pred)-1):
    if y_pred[i] == 1:
        check.append(y_pred[i] == classification_target.to_numpy()[i])

In [26]:
print(signals)
print(check)
print(record_1)
print(record_2)

[(1.0, 0.120000000000001), (1.0, 0.1899999999999995), (1.0, 0.120000000000001), (1.0, 0.07000000000000028), (1.0, 0.1899999999999995), (1.0, 0.14999999999999947), (1.0, 0.020000000000000462), (0.0, -0.20999999999999996), (1.0, 0.05000000000000071), (1.0, 0.040000000000000036), (1.0, 0.02999999999999936), (1.0, 0.2400000000000002), (0.0, 0.0), (0.0, 0.0), (1.0, 0.04999999999999982), (1.0, 0.040000000000000036), (1.0, 0.03000000000000025), (1.0, 0.0699999999999994), (0.0, 0.0), (1.0, 0.5199999999999996), (1.0, 0.1900000000000004), (0.0, 0.0), (1.0, 0.019999999999999574), (0.0, -0.1900000000000004), (1.0, 0.04999999999999982), (0.0, -0.27999999999999936), (1.0, 0.019999999999999574), (1.0, 0.1200000000000001), (1.0, 0.09999999999999964), (0.0, -0.020000000000000462), (0.0, -0.08000000000000007), (1.0, 0.08999999999999986), (1.0, 0.040000000000000036), (0.0, -0.1200000000000001), (0.0, -0.08000000000000007), (1.0, 0.2400000000000002), (1.0, 0.23000000000000043), (1.0, 0.09999999999999964),

In [27]:
for i, j in zip(record_1, record_2):
    if i != j:
        print(i, j)


## 👋 Chào mừng bạn đến với Vnstock!

Cảm ơn bạn đã sử dụng package phân tích chứng khoán #1 tại Việt Nam

* Tài liệu: [Sổ tay hướng dẫn](https://vnstocks.com/docs/category/s%E1%BB%95-tay-h%C6%B0%E1%BB%9Bng-d%E1%BA%ABn)
* Cộng đồng: [Nhóm Facebook](https://www.facebook.com/groups/vnstock.official)

Khám phá các tính năng mới nhất và tham gia cộng đồng để nhận hỗ trợ.
                


## 👋 Chào mừng bạn đến với Vnstock!

Cảm ơn bạn đã sử dụng package phân tích chứng khoán #1 tại Việt Nam

* Tài liệu: [Sổ tay hướng dẫn](https://vnstocks.com/docs/category/s%E1%BB%95-tay-h%C6%B0%E1%BB%9Bng-d%E1%BA%ABn)
* Cộng đồng: [Nhóm Facebook](https://www.facebook.com/groups/vnstock.official)

Khám phá các tính năng mới nhất và tham gia cộng đồng để nhận hỗ trợ.
                


## 👋 Chào mừng bạn đến với Vnstock!

Cảm ơn bạn đã sử dụng package phân tích chứng khoán #1 tại Việt Nam

* Tài liệu: [Sổ tay hướng dẫn](https://vnstocks.com/docs/category/s%E1%BB%95-tay-h%C6%B0%E1%BB%9Bng-d%E1%BA%ABn)
* Cộng đồng: [Nhóm Facebook](https://www.facebook.com/groups/vnstock.official)

Khám phá các tính năng mới nhất và tham gia cộng đồng để nhận hỗ trợ.
                


## 👋 Chào mừng bạn đến với Vnstock!

Cảm ơn bạn đã sử dụng package phân tích chứng khoán #1 tại Việt Nam

* Tài liệu: [Sổ tay hướng dẫn](https://vnstocks.com/docs/category/s%E1%BB%95-tay-h%C6%B0%E1%BB%9Bng-d%E1%BA%ABn)
* Cộng đồng: [Nhóm Facebook](https://www.facebook.com/groups/vnstock.official)

Khám phá các tính năng mới nhất và tham gia cộng đồng để nhận hỗ trợ.
                


## 👋 Chào mừng bạn đến với Vnstock!

Cảm ơn bạn đã sử dụng package phân tích chứng khoán #1 tại Việt Nam

* Tài liệu: [Sổ tay hướng dẫn](https://vnstocks.com/docs/category/s%E1%BB%95-tay-h%C6%B0%E1%BB%9Bng-d%E1%BA%ABn)
* Cộng đồng: [Nhóm Facebook](https://www.facebook.com/groups/vnstock.official)

Khám phá các tính năng mới nhất và tham gia cộng đồng để nhận hỗ trợ.
                


## 👋 Chào mừng bạn đến với Vnstock!

Cảm ơn bạn đã sử dụng package phân tích chứng khoán #1 tại Việt Nam

* Tài liệu: [Sổ tay hướng dẫn](https://vnstocks.com/docs/category/s%E1%BB%95-tay-h%C6%B0%E1%BB%9Bng-d%E1%BA%ABn)
* Cộng đồng: [Nhóm Facebook](https://www.facebook.com/groups/vnstock.official)

Khám phá các tính năng mới nhất và tham gia cộng đồng để nhận hỗ trợ.
                